In [ ]:
import asyncio
import json
import os
import imaplib
import email
import PyPDF2
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# =====================================================================
# 1. TUS CREDENCIALES
# =====================================================================
CORREO_GMAIL = ""
PASSWORD_APP = ""
API_KEY_GROQ = ""

# =====================================================================
# 2. EL ERP SIMULADO
# =====================================================================
ERP_ORDENES_COMPRA = {
    "OC-2026-002": {"rfc_proveedor": "RFC-OFFICE-456", "monto_autorizado": 15000.00}
}
ERP_RECEPCION_ALMACEN = {
    "OC-2026-002": {"mercancia_recibida": True}
}

# =====================================================================
# 3. LA CAJA Y EL CEREBRO
# =====================================================================
class EstadoAuditoria(TypedDict):
    texto_pdf_crudo: str
    proveedor: str
    rfc: str
    monto_total: float
    orden_compra_referencia: str
    estatus_validacion: str
    mensaje_final: str

llm_json = ChatGroq(
    model_name="llama-3.1-8b-instant", 
    groq_api_key=API_KEY_GROQ, 
    temperature=0, 
    model_kwargs={"response_format": {"type": "json_object"}}
)

# =====================================================================
# 4. FUNCIONES DE DESCARGA Y LECTURA (SÍNCRONAS)
# =====================================================================
def entrar_al_correo_y_leer_pdf():
    print(f"Conectando al correo: {CORREO_GMAIL}")
    texto_extraido = ""
    try: 
        mail = imaplib.IMAP4_SSL('imap.gmail.com')
        mail.login(CORREO_GMAIL, PASSWORD_APP)
        mail.select('inbox')
        
        status, mensajes = mail.search(None, '(SUBJECT "Factura" UNSEEN)')
        ids_correos = mensajes[0].split()
        
        if not ids_correos:
            print("No hay correos nuevos con el asunto 'Factura'.")
            return None
            
        ultimo_id = ids_correos[-1]
        status, datos_correo = mail.fetch(ultimo_id, '(RFC822)')

        for response_part in datos_correo:
            if isinstance(response_part, tuple):
                mensaje = email.message_from_bytes(response_part[1])

                for part in mensaje.walk():
                    if part.get_content_maintype() == 'multipart' or part.get('Content-Disposition') is None:
                        continue

                    nombre_archivo = part.get_filename()
                    if nombre_archivo and nombre_archivo.lower().endswith('.pdf'):
                        ruta_guardado = os.path.join(os.getcwd(), nombre_archivo)

                        with open(ruta_guardado, 'wb') as f:
                            f.write(part.get_payload(decode=True))
                        print(f"PDF descargado: {nombre_archivo}")

                        with open(ruta_guardado, 'rb') as archivo_pdf:
                            lector = PyPDF2.PdfReader(archivo_pdf)
                            for pagina in lector.pages:
                                texto_extraido += pagina.extract_text() + "\n"

                        print(f"✅ Texto raspado con éxito ({len(texto_extraido)} caracteres).")
                        return texto_extraido
        
        print("❌ Encontré el correo, pero no traía PDF.")
        return None
        
    except Exception as e:
        print(f"❌ Error en el correo: {e}")
        return None

# =====================================================================
# 5. ESTACIONES DE TRABAJO
# =====================================================================
async def nodo_ingesta(state: EstadoAuditoria):
    texto = await asyncio.to_thread(entrar_al_correo_y_leer_pdf)

    if not texto:
        return {"estatus_validacion": "ERROR_CORREO", "mensaje_final": "Proceso abortado en la ingesta"}
    return {"texto_pdf_crudo": texto}

async def nodo_extractor(state: EstadoAuditoria):
    print("[Extractor IA] Mandando texto a Groq")
    texto = state["texto_pdf_crudo"]

    prompt = f"""
    Extrae la información de este texto sacado de una factura.
    
    Texto:
    "{texto}"
    
    Instrucciones:
    1. "proveedor": Nombre de la empresa.
    2. "rfc": Registro fiscal.
    3. "monto_total": Número decimal final (float).
    4. "orden_compra": Código tipo OC-XXXX (si no hay, deja vacío).
    
    Responde ÚNICAMENTE en JSON:
    {{
        "proveedor": "nombre",
        "rfc": "rfc",
        "monto_total": 0.00,
        "orden_compra": "codigo"
    }}
    """
    respuesta = await llm_json.ainvoke(prompt)
    datos = json.loads(respuesta.content)

    print(f"Groq extrajo -> Prov: {datos.get('proveedor')} | OC: {datos.get('orden_compra')} | Total: ${datos.get('monto_total')}")

    return {
        "proveedor": datos.get("proveedor"),
        "rfc": datos.get("rfc"),
        "monto_total": float(datos.get("monto_total", 0.0)),
        "orden_compra_referencia": datos.get("orden_compra")
    }

async def nodo_validador(state: EstadoAuditoria):
    print("[Validador] Cruzando con la base de datos")
    oc_id = state.get("orden_compra_referencia")
    monto = state.get("monto_total")

    if oc_id not in ERP_ORDENES_COMPRA:
        return {"estatus_validacion": "RECHAZO", "mensaje_final": f"La OC '{oc_id}' no existe en el sistema."}
    
    oc_sistema = ERP_ORDENES_COMPRA[oc_id]

    if monto != oc_sistema["monto_autorizado"]:
        return {"estatus_validacion": "RECHAZO", "mensaje_final": f"❌ Fraude/Error: Cobran ${monto} pero se autorizaron ${oc_sistema['monto_autorizado']}."}

    if not ERP_RECEPCION_ALMACEN.get(oc_id, {}).get("mercancia_recibida"):
        return {"estatus_validacion": "RECHAZO", "mensaje_final": "⏳ Almacén aún no recibe la mercancía."}
        
    return {"estatus_validacion": "APROBADO", "mensaje_final": "🎉 FACTURA APROBADA y lista para pago."}

def enrutador_fallas(state: EstadoAuditoria) -> Literal["nodo_extractor", "__end__"]:
    if state.get("estatus_validacion") == "ERROR_CORREO":
        return END
    return "nodo_extractor"

# =====================================================================
# 6. ENSAMBLE DEL GRAFO
# =====================================================================
flujo = StateGraph(EstadoAuditoria)
flujo.add_node("nodo_ingesta", nodo_ingesta)
flujo.add_node("nodo_extractor", nodo_extractor)
flujo.add_node("nodo_validador", nodo_validador)

flujo.add_edge(START, "nodo_ingesta")
flujo.add_conditional_edges("nodo_ingesta", enrutador_fallas)
flujo.add_edge("nodo_extractor", "nodo_validador")
flujo.add_edge("nodo_validador", END)

agente_real = flujo.compile()

# =====================================================================
# 7. EJECUCIÓN
# =====================================================================
async def arrancar():
    print("Iniciando automatizacion")
    caja_inicial = {
        "texto_pdf_crudo": "", "proveedor": "",
        "rfc": "", "monto_total": 0.0, "orden_compra_referencia": "",
        "estatus_validacion": "", "mensaje_final": ""
    }

    resultado = await agente_real.ainvoke(caja_inicial)
    print("\n================ RESULTADO FINAL ================")
    print(resultado["mensaje_final"])

await arrancar()